# Imports

In [26]:
## load packages 
import pandas as pd
import re
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

## nltk imports
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## random
import random

In [27]:
#pull 2020 data
df2020 = nj_censustracts_2020 = pd.read_csv("../data/DECENNIALDP2020.DP1-2026-05-14T152641(2).csv")

In [28]:
df2020.head()

,census_tract,Census Tract 1.01; Hudson County; New Jersey!!Count,Census Tract 1.01; Hudson County; New Jersey!!Percent,Census Tract 1.02; Hudson County; New Jersey!!Count,Census Tract 1.02; Hudson County; New Jersey!!Percent,Census Tract 2; Hudson County; New Jersey!!Count,Census Tract 2; Hudson County; New Jersey!!Percent,Census Tract 3; Hudson County; New Jersey!!Count,Census Tract 3; Hudson County; New Jersey!!Percent,Census Tract 4; Hudson County; New Jersey!!Count,...,Census Tract 199; Hudson County; New Jersey!!Count,Census Tract 199; Hudson County; New Jersey!!Percent,Census Tract 200; Hudson County; New Jersey!!Count,Census Tract 200; Hudson County; New Jersey!!Percent,Census Tract 201; Hudson County; New Jersey!!Count,Census Tract 201; Hudson County; New Jersey!!Percent,Census Tract 324; Hudson County; New Jersey!!Count,Census Tract 324; Hudson County; New Jersey!!Percent,Census Tract 9801; Hudson County; New Jersey!!Count,Census Tract 9801; Hudson County; New Jersey!!Percent
0,total_pop,"2,554",100.00%,"3,834",100.00%,"5,391",100.00%,"3,948",100.00%,"3,973",...,"5,542",100.00%,"5,303",100.00%,"4,256",100.00%,"6,762",100.00%,5.0,100.00%
1,under_5,160,6.30%,211,5.50%,273,5.10%,198,5.00%,247,...,335,6.00%,190,3.60%,469,11.00%,382,5.60%,1.0,20.00%
2,5_to_9,161,6.30%,205,5.30%,345,6.40%,215,5.40%,258,...,251,4.50%,187,3.50%,197,4.60%,440,6.50%,0.0,0.00%
3,10 to 14 years,171,6.70%,215,5.60%,301,5.60%,201,5.10%,205,...,258,4.70%,271,5.10%,115,2.70%,477,7.10%,1.0,20.00%
4,15 to 19 years,146,5.70%,211,5.50%,282,5.20%,185,4.70%,196,...,253,4.60%,310,5.80%,83,2.00%,450,6.70%,0.0,0.00%


In [29]:
#transpose the data so census tracts are the rows
df2020 = df2020.transpose()

In [30]:
df2020.head()

,0,1,2,3,4,5,6,7,8,9,...,163,164,165,166,167,168,169,170,171,172
census_tract,total_pop,under_5,5_to_9,10 to 14 years,15 to 19 years,20 to 24 years,25 to 29 years,30 to 34 years,35 to 39 years,40 to 44 years,...,"Sold, not occupied","For seasonal, recreational, or occ...",All other vacants,VACANCY RATES,Homeowner vacancy rate (percent) [4],Rental vacancy rate (percent) [5],HOUSING TENURE,Occupied housing units,Owner-occupied housing units,Renter-occupied housing units
Census Tract 1.01; Hudson County; New Jersey!!Count,"2,554",160,161,171,146,173,265,247,213,149,...,6,7,27,NaN,3,4.4,NaN,847,321,526
Census Tract 1.01; Hudson County; New Jersey!!Percent,100.00%,6.30%,6.30%,6.70%,5.70%,6.80%,10.40%,9.70%,8.30%,5.80%,...,0.70%,0.80%,2.90%,NaN,(X),(X),NaN,100.00%,37.90%,62.10%
Census Tract 1.02; Hudson County; New Jersey!!Count,"3,834",211,205,215,211,271,350,426,322,245,...,2,3,39,NaN,1.4,5.5,NaN,"1,330",433,897
Census Tract 1.02; Hudson County; New Jersey!!Percent,100.00%,5.50%,5.30%,5.60%,5.50%,7.10%,9.10%,11.10%,8.40%,6.40%,...,0.10%,0.20%,2.70%,NaN,(X),(X),NaN,100.00%,32.60%,67.40%


In [31]:
#correcting indices
df2020 = df2020.reset_index()
df2020.columns = df2020.iloc[0]

In [32]:
df2020.head()

,census_tract,total_pop,under_5,5_to_9,10 to 14 years,15 to 19 years,20 to 24 years,25 to 29 years,30 to 34 years,35 to 39 years,...,"Sold, not occupied","For seasonal, recreational, or occasional use",All other vacants,VACANCY RATES,Homeowner vacancy rate (percent) [4],Rental vacancy rate (percent) [5],HOUSING TENURE,Occupied housing units,Owner-occupied housing units,Renter-occupied housing units
0,census_tract,total_pop,under_5,5_to_9,10 to 14 years,15 to 19 years,20 to 24 years,25 to 29 years,30 to 34 years,35 to 39 years,...,"Sold, not occupied","For seasonal, recreational, or occ...",All other vacants,VACANCY RATES,Homeowner vacancy rate (percent) [4],Rental vacancy rate (percent) [5],HOUSING TENURE,Occupied housing units,Owner-occupied housing units,Renter-occupied housing units
1,Census Tract 1.01; Hudson County; New Jersey!!...,"2,554",160,161,171,146,173,265,247,213,...,6,7,27,NaN,3,4.4,NaN,847,321,526
2,Census Tract 1.01; Hudson County; New Jersey!!...,100.00%,6.30%,6.30%,6.70%,5.70%,6.80%,10.40%,9.70%,8.30%,...,0.70%,0.80%,2.90%,NaN,(X),(X),NaN,100.00%,37.90%,62.10%
3,Census Tract 1.02; Hudson County; New Jersey!!...,"3,834",211,205,215,211,271,350,426,322,...,2,3,39,NaN,1.4,5.5,NaN,"1,330",433,897
4,Census Tract 1.02; Hudson County; New Jersey!!...,100.00%,5.50%,5.30%,5.60%,5.50%,7.10%,9.10%,11.10%,8.40%,...,0.10%,0.20%,2.70%,NaN,(X),(X),NaN,100.00%,32.60%,67.40%


In [33]:
#Eliminating rows that count demographic totals. Only keeping rows that count percentages
df2020 = df2020[df2020.census_tract.str.contains("!!Percent")]
df2020.head()

,census_tract,total_pop,under_5,5_to_9,10 to 14 years,15 to 19 years,20 to 24 years,25 to 29 years,30 to 34 years,35 to 39 years,...,"Sold, not occupied","For seasonal, recreational, or occasional use",All other vacants,VACANCY RATES,Homeowner vacancy rate (percent) [4],Rental vacancy rate (percent) [5],HOUSING TENURE,Occupied housing units,Owner-occupied housing units,Renter-occupied housing units
2,Census Tract 1.01; Hudson County; New Jersey!!...,100.00%,6.30%,6.30%,6.70%,5.70%,6.80%,10.40%,9.70%,8.30%,...,0.70%,0.80%,2.90%,NaN,(X),(X),NaN,100.00%,37.90%,62.10%
4,Census Tract 1.02; Hudson County; New Jersey!!...,100.00%,5.50%,5.30%,5.60%,5.50%,7.10%,9.10%,11.10%,8.40%,...,0.10%,0.20%,2.70%,NaN,(X),(X),NaN,100.00%,32.60%,67.40%
6,Census Tract 2; Hudson County; New Jersey!!Per...,100.00%,5.10%,6.40%,5.60%,5.20%,7.30%,9.60%,10.40%,8.10%,...,0.20%,0.00%,2.40%,NaN,(X),(X),NaN,100.00%,25.00%,75.00%
8,Census Tract 3; Hudson County; New Jersey!!Per...,100.00%,5.00%,5.40%,5.10%,4.70%,7.40%,12.10%,10.60%,9.70%,...,0.60%,0.60%,3.20%,NaN,(X),(X),NaN,100.00%,31.30%,68.70%
10,Census Tract 4; Hudson County; New Jersey!!Per...,100.00%,6.20%,6.50%,5.20%,4.90%,7.00%,11.20%,9.50%,7.80%,...,0.40%,0.00%,3.70%,NaN,(X),(X),NaN,100.00%,31.90%,68.10%


In [34]:
#Creating boolean variable near_hblr and listing all census tracts that count as near_hblr
keywords = [' 73;', ' 75;', ' 74;', ' 76.01;', ' 76.02;', ' 77.01;', ' 77.02;', ' 77.03;', ' 193;', ' 194;', ' 115;', ' 114;', ' 113;', ' 112;', ' 110;', ' 109;', ' 107.01;', ' 107.02;', ' 104;', ' 103;', ' 102;', ' 63;', ' 62;', ' 60;', ' 58.01;', ' 55;', ' 53;', ' 68;', ' 44;', ' 45;', ' 46;', ' 47;', ' 48;', ' 49;', ' 42;', ' 190;', ' 191;', ' 192;', ' 3;', ' 8;', ' 189;', ' 185.01;', ' 185.02;', ' 179;', ' 146;', ' 160;', ' 162;', ' 158.02;', ' 161;']
df2020.loc[:, 'near_hblr'] = df2020['census_tract'].str.contains('|'.join(keywords), na=False)


In [35]:
df2020.head()

,census_tract,total_pop,under_5,5_to_9,10 to 14 years,15 to 19 years,20 to 24 years,25 to 29 years,30 to 34 years,35 to 39 years,...,"For seasonal, recreational, or occasional use",All other vacants,VACANCY RATES,Homeowner vacancy rate (percent) [4],Rental vacancy rate (percent) [5],HOUSING TENURE,Occupied housing units,Owner-occupied housing units,Renter-occupied housing units,near_hblr
2,Census Tract 1.01; Hudson County; New Jersey!!...,100.00%,6.30%,6.30%,6.70%,5.70%,6.80%,10.40%,9.70%,8.30%,...,0.80%,2.90%,NaN,(X),(X),NaN,100.00%,37.90%,62.10%,False
4,Census Tract 1.02; Hudson County; New Jersey!!...,100.00%,5.50%,5.30%,5.60%,5.50%,7.10%,9.10%,11.10%,8.40%,...,0.20%,2.70%,NaN,(X),(X),NaN,100.00%,32.60%,67.40%,False
6,Census Tract 2; Hudson County; New Jersey!!Per...,100.00%,5.10%,6.40%,5.60%,5.20%,7.30%,9.60%,10.40%,8.10%,...,0.00%,2.40%,NaN,(X),(X),NaN,100.00%,25.00%,75.00%,False
8,Census Tract 3; Hudson County; New Jersey!!Per...,100.00%,5.00%,5.40%,5.10%,4.70%,7.40%,12.10%,10.60%,9.70%,...,0.60%,3.20%,NaN,(X),(X),NaN,100.00%,31.30%,68.70%,True
10,Census Tract 4; Hudson County; New Jersey!!Per...,100.00%,6.20%,6.50%,5.20%,4.90%,7.00%,11.20%,9.50%,7.80%,...,0.00%,3.70%,NaN,(X),(X),NaN,100.00%,31.90%,68.10%,False


In [36]:
#checking column names
print(df2020.columns.tolist())

['census_tract', 'total_pop', 'under_5', '5_to_9', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa010 to 14 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa015 to 19 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa020 to 24 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa025 to 29 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa030 to 34 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa035 to 39 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa040 to 44 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa045 to 49 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa050 to 54 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa055 to 59 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa060 to 64 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa065 to 69 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa070 to 74 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa075 to 79 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa080 to 84 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa085 years and over', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Selected Age Categories', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa016 years and over', '\xa0\xa0\xa0\xa0\

In [37]:
#cleaning column names
df2020.columns = df2020.columns.str.replace('\xa0', '', regex=False)

In [38]:
print(df2020.columns.tolist())

['census_tract', 'total_pop', 'under_5', '5_to_9', '10 to 14 years', '15 to 19 years', '20 to 24 years', '25 to 29 years', '30 to 34 years', '35 to 39 years', '40 to 44 years', '45 to 49 years', '50 to 54 years', '55 to 59 years', '60 to 64 years', '65 to 69 years', '70 to 74 years', '75 to 79 years', '80 to 84 years', '85 years and over', 'Selected Age Categories', '16 years and over', '18 years and over', '21 years and over', '62 years and over', '65 years and over', 'Male population', 'Under 5 years', '5 to 9 years', '10 to 14 years', '15 to 19 years', '20 to 24 years', '25 to 29 years', '30 to 34 years', '35 to 39 years', '40 to 44 years', '45 to 49 years', '50 to 54 years', '55 to 59 years', '60 to 64 years', '65 to 69 years', '70 to 74 years', '75 to 79 years', '80 to 84 years', '85 years and over', 'Selected Age Categories', '16 years and over', '18 years and over', '21 years and over', '62 years and over', '65 years and over', 'Female population', 'Under 5 years', '5 to 9 years

In [39]:
#giving repeat column names distinct labels
cols = list(df2020.columns)
seen = {}
for i, col in enumerate(cols):
    if col in seen:
        seen[col] += 1
        cols[i] = f"{col}_{seen[col]}"
    else:
        seen[col] = 0
df2020.columns = cols

In [40]:
#creating racial percentages
df2020['white_pct'] = df2020['White alone_1'].str.replace('%', '', regex=False).astype(float)
df2020['black_pct'] = df2020['Black or African American alone_1'].str.replace('%', '', regex=False).astype(float)
df2020['asian_pct'] = df2020['Asian alone_1'].str.replace('%', '', regex=False).astype(float)
df2020['amerindian_pct'] = df2020['American Indian and Alaska Native alone_1'].str.replace('%', '', regex=False).astype(float)
df2020['other_pct'] = df2020['Some Other Race alone_1'].str.replace('%', '', regex=False).astype(float)

In [41]:
#creating hispanic race percentage
df2020['hispanic_pct'] = df2020['Hispanic or Latino'].str.replace('%', '', regex=False).astype(float)


In [42]:
#checking new columns
df2020[['census_tract', 'white_pct', 'black_pct', 'asian_pct', 'other_pct', 'hispanic_pct', 'near_hblr']].head(20)

,census_tract,white_pct,black_pct,asian_pct,other_pct,hispanic_pct,near_hblr
2,Census Tract 1.01; Hudson County; New Jersey!!...,17.8,4.1,33.8,1.4,41.5,False
4,Census Tract 1.02; Hudson County; New Jersey!!...,18.3,5.7,32.2,0.8,41.5,False
6,Census Tract 2; Hudson County; New Jersey!!Per...,15.9,5.9,19.3,1.6,54.8,False
8,Census Tract 3; Hudson County; New Jersey!!Per...,31.6,3.6,12.6,1.1,48.4,True
10,Census Tract 4; Hudson County; New Jersey!!Per...,16.8,3.7,41.6,1.2,34.5,False
12,Census Tract 5; Hudson County; New Jersey!!Per...,22.2,4.6,30.3,1.3,38.8,False
14,Census Tract 6; Hudson County; New Jersey!!Per...,21.4,4.9,27.7,1.3,42.3,False
16,Census Tract 7; Hudson County; New Jersey!!Per...,26.5,4.7,13.5,1.3,51.6,False
18,Census Tract 8; Hudson County; New Jersey!!Per...,38.8,4.4,13.3,1.7,38.3,True
20,Census Tract 9.02; Hudson County; New Jersey!!...,19.8,4.1,60.1,0.9,13.2,False


In [43]:
df2020.groupby("near_hblr")[["white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].mean().reset_index()

,near_hblr,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
0,False,27.426866,7.290299,16.789552,43.713433,1.343284
1,True,32.451020,16.889796,16.328571,30.244898,1.042857


In [44]:
#pull 2010 data
df2010 = pd.read_csv("../data/DECENNIALSF12010.P9-2026-05-19T020157 (2010).csv")

In [45]:
#transpose the data so census tracts are the rows
df2010 = df2010.transpose()

In [46]:
#correct indices
df2010.columns = df2010.iloc[0]
df2010 = df2010.reset_index()

In [47]:
df2010.head()

Label (Grouping),index,Total:,Hispanic or Latino,White alone,Black or African American alone,American Indian and Alaska Native alone,Asian alone,Native Hawaiian and Other Pacific Islander alone,Some Other Race alone,Two or More Races:
0,Label (Grouping),Total:,Hispanic or Latino,White alone,Black or African American alone,American Indian and Alaska Native ...,Asian alone,Native Hawaiian and Other Pacific ...,Some Other Race alone,Two or More Races:
1,"Census Tract 1, Hudson County, New Jersey","6,025","2,458","1,437",253,15,"1,693",9,32,128
2,"Census Tract 2, Hudson County, New Jersey","5,409","3,362",987,310,12,630,0,31,77
3,"Census Tract 3, Hudson County, New Jersey","4,220 (r46394)","2,346","1,267",177,10,312,0,35,73
4,"Census Tract 4, Hudson County, New Jersey","3,991","1,297","1,013",205,10,"1,297",0,39,130


In [48]:
#clean column names
df2010.columns = df2010.columns.str.replace('\xa0', '', regex=False)
print(df2010.columns.tolist())

['index', 'Total:', 'Hispanic or Latino', 'White alone', 'Black or African American alone', 'American Indian and Alaska Native alone', 'Asian alone', 'Native Hawaiian and Other Pacific Islander alone', 'Some Other Race alone', 'Two or More Races:']


In [49]:
df2010 = df2010.drop(0)

In [50]:
#converting string values in columns to floats
df2010['Total:'] = df2010['Total:'].str.replace(r'\s*\(.*?\)', '', regex=True)
df2010['Total:'] = df2010['Total:'].str.replace(',', '', regex=False).astype(float)
df2010['White alone'] = df2010['White alone'].str.replace(',', '', regex=False).astype(float)
df2010['Black or African American alone'] = df2010['Black or African American alone'].str.replace(',', '', regex=False).astype(float)
df2010['Asian alone'] = df2010['Asian alone'].str.replace(',', '', regex=False).astype(float)
df2010['Some Other Race alone'] = df2010['Some Other Race alone'].str.replace(',', '', regex=False).astype(float)
df2010['Hispanic or Latino'] = df2010['Hispanic or Latino'].str.replace(',', '', regex=False).astype(float)

In [51]:
df2010['white_pct'] = df2010['White alone'] / df2010['Total:'] * 100
df2010['black_pct'] = df2010['Black or African American alone'] / df2010['Total:'] * 100
df2010['asian_pct'] = df2010['Asian alone'] / df2010['Total:'] * 100
df2010['other_pct'] = df2010['Some Other Race alone'] / df2010['Total:'] * 100
df2010['hispanic_pct'] = df2010['Hispanic or Latino'] / df2010['Total:'] * 100

In [52]:
#coding same near_hblr boolean variable for 2010 dataset as for 2020 dataset
keywords = [' 73,', ' 75,', ' 74,', ' 76.01,', ' 76.02,', ' 77.01,', ' 77.02,', ' 77.03,', ' 193,', ' 194,', ' 115,', ' 114,', ' 113,', ' 112,', ' 110,', ' 109,', ' 107.01,', ' 107.02,', ' 104,', ' 103,', ' 102,', ' 63,', ' 62,', ' 60,', ' 58.01,', ' 55,', ' 53,', ' 68,', ' 44,', ' 45,', ' 46,', ' 47,', ' 48,', ' 49,', ' 42,', ' 190,', ' 191,', ' 192,', ' 3,', ' 8,', ' 189,', ' 185.01,', ' 185.02,', ' 179,', ' 146,', ' 160,', ' 162,', ' 158.02,', ' 161,']
df2010.loc[:, 'near_hblr'] = df2010['index'].str.contains('|'.join(keywords), na=False)

In [53]:
df2010[["near_hblr", "white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].head()

Label (Grouping),near_hblr,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
1,False,23.850622,4.199170,28.099585,40.796680,0.531120
2,False,18.247366,5.731189,11.647255,62.155666,0.573119
3,True,30.023697,4.194313,7.393365,55.592417,0.829384
4,False,25.382110,5.136557,32.498121,32.498121,0.977199
5,False,22.662955,5.196010,24.634656,44.722802,1.113431


In [54]:
#coding near_path boolean variable for both 2010 and 2020 datasets based on which tracts are near path
keywords = [' 12.02,', ' 9.02,', ' 71,', ' 19,', ' 20.02,', ' 20.01,', ' 70.02,', ' 70.01,', ' 64,', ' 35,', ' 75,', ' 74,', ' 76.01,', ' 76.02,', ' 77.01,', ' 77.02,', ' 77.03,', ' 193,', ' 194,', ]
df2010.loc[:, 'near_path'] = df2010['index'].str.contains('|'.join(keywords), na=False)
keywords = [' 12.02;', ' 9.02;', ' 71;', ' 19;', ' 20.02;', ' 20.01;', ' 70.02;', ' 70.01;', ' 64;', ' 35;', ' 75;', ' 74;', ' 76.01;', ' 76.02;', ' 77.01;', ' 77.02;', ' 77.03;', ' 193;', ' 194;', ]
df2020.loc[:, 'near_path'] = df2020['census_tract'].str.contains('|'.join(keywords), na=False)

In [55]:
df2010[~df2010['near_path']].groupby("near_hblr")[["white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].mean().reset_index()

Label (Grouping),near_hblr,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
0,False,31.831680,7.733542,13.593792,44.401811,0.674379
1,True,30.328015,25.319812,7.396210,34.492046,0.671600


In [56]:
df2020[~df2020['near_path']].groupby("near_hblr")[["white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].mean().reset_index()

,near_hblr,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
0,False,27.066935,7.241935,15.410484,45.500806,1.378226
1,True,30.867500,20.135000,9.637500,35.207500,1.107500


In [57]:
df2010.groupby(["near_hblr", "near_path"])[["white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].mean().reset_index()

df2020.groupby(["near_hblr", "near_path"])[["white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].mean().reset_index()

Label (Grouping),near_hblr,near_path,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
0,False,False,31.831680,7.733542,13.593792,44.401811,0.674379
1,False,True,30.949693,10.133999,30.464003,24.814023,0.700023
2,True,False,30.328015,25.319812,7.396210,34.492046,0.671600
3,True,True,61.004618,3.610017,20.497808,12.390290,0.294934


,near_hblr,near_path,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
0,False,False,27.066935,7.241935,15.410484,45.500806,1.378226
1,False,True,31.890000,7.890000,33.890000,21.550000,0.910000
2,True,False,30.867500,20.135000,9.637500,35.207500,1.107500
3,True,True,39.488889,2.466667,46.066667,8.188889,0.755556


In [58]:
df2020.to_csv('../data/2020cleaned.csv', index=False)
df2010.to_csv('../data/2010cleaned.csv', index=False)